# Loss Landscape and Contour Explorer
Move the horizontal plane and see the matching contour level on the same lecture-style loss surface and GD trajectory.


### Optional dependency setup
Run this cell only if imports fail in your environment.

In [ ]:
import importlib.util

packages = [
    'numpy', 'matplotlib', 'plotly', 'ipywidgets', 'scipy', 'pandas', 'sklearn', 'seaborn'
]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', missing)
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
else:
    print('All common packages already available.')


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display


def L(t0, t1):
    return np.sin(t0) * np.cos(t1) + 0.1 * (t0 * t0 + t1 * t1)


def grad(t0, t1):
    g0 = np.cos(t0) * np.cos(t1) + 0.2 * t0
    g1 = -np.sin(t0) * np.sin(t1) + 0.2 * t1
    return g0, g1


lo, hi = -5.0, 5.0
xs = np.linspace(lo, hi, 100)
ys = np.linspace(lo, hi, 100)
X, Y = np.meshgrid(xs, ys)
Z = L(X, Y)

out = widgets.Output()
plane = widgets.FloatSlider(description='plane', min=-1.0, max=5.5, step=0.05, value=1.2, readout_format='.2f', continuous_update=False)
alpha = widgets.FloatSlider(description='alpha', min=0.005, max=0.20, step=0.005, value=0.10, readout_format='.3f', continuous_update=False)
max_iter = widgets.IntSlider(description='max iter', min=1, max=200, step=1, value=40, continuous_update=False)


def gd_path(a, steps):
    t0, t1 = 4.5, -4.5
    px, py, pz = [t0], [t1], [float(L(t0, t1))]
    for _ in range(steps):
        g0, g1 = grad(t0, t1)
        t0 -= a * g0
        t1 -= a * g1
        px.append(float(t0))
        py.append(float(t1))
        pz.append(float(L(t0, t1)))
    return np.array(px), np.array(py), np.array(pz)


def render(*_):
    px, py, pz = gd_path(alpha.value, max_iter.value)

    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{'type': 'scene'}, {'type': 'xy'}]],
        subplot_titles=('Loss Landscape', 'Contour Plot with Gradient Descent Trajectory'),
        horizontal_spacing=0.08,
    )

    fig.add_trace(go.Surface(x=xs, y=ys, z=Z, colorscale='Viridis', showscale=False, opacity=0.85, name='loss'), row=1, col=1)
    fig.add_trace(
        go.Surface(
            x=xs,
            y=ys,
            z=np.full_like(Z, plane.value),
            showscale=False,
            opacity=0.28,
            colorscale=[[0, '#f97316'], [1, '#f97316']],
            name='selected plane',
        ),
        row=1,
        col=1,
    )
    fig.add_trace(go.Scatter3d(x=px, y=py, z=pz, mode='lines', line=dict(color='#dc2626', width=6), name='GD trajectory'), row=1, col=1)

    fig.add_trace(
        go.Contour(
            x=xs,
            y=ys,
            z=Z,
            colorscale='Viridis',
            showscale=False,
            contours=dict(coloring='lines', showlabels=True, labelfont=dict(size=10, color='#334155')),
            line=dict(width=1),
            name='contour levels',
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Contour(
            x=xs,
            y=ys,
            z=Z,
            showscale=False,
            line=dict(color='#f97316', width=3),
            contours=dict(start=plane.value, end=plane.value, size=1, coloring='lines'),
            name='plane contour',
        ),
        row=1,
        col=2,
    )
    fig.add_trace(go.Scatter(x=px, y=py, mode='lines', line=dict(color='#dc2626', width=3), name='GD path'), row=1, col=2)

    fig.update_scenes(
        xaxis_title='θ0',
        yaxis_title='θ1',
        zaxis_title='L(θ)',
        xaxis=dict(range=[lo, hi]),
        yaxis=dict(range=[lo, hi]),
        zaxis=dict(range=[-1.2, 5.5]),
        camera=dict(eye=dict(x=1.45, y=-1.8, z=1.1)),
        row=1,
        col=1,
    )
    fig.update_xaxes(title_text='θ0', range=[lo, hi], row=1, col=2)
    fig.update_yaxes(title_text='θ1', range=[lo, hi], scaleanchor='x', row=1, col=2)
    fig.update_layout(height=520, showlegend=False)

    with out:
        out.clear_output(wait=True)
        fig.show()
        print(
            f'selected level={plane.value:.2f} | alpha={alpha.value:.3f} | iter={max_iter.value} '
            f'| start theta=(4.500, -4.500) | final theta=({px[-1]:.3f}, {py[-1]:.3f}) | final loss={pz[-1]:.3f}'
        )


for w in [plane, alpha, max_iter]:
    w.observe(render, 'value')

display(widgets.HBox([plane, alpha, max_iter]))
display(out)
render()
